# Particle-1 distillation: Qwen3-1.7B QLoRA -> Q4_0 GGUF

Kaggle setup:
1. Settings -> Accelerator -> **GPU T4 x2**
2. Add the repo (or upload `data/` from `python3 distill/build_dataset.py`) as a Kaggle Dataset
3. Run all; download `particle1-qwen3-1.7b-Q4_0.gguf` from the output

Student: `Qwen/Qwen3-1.7B`. Teacher traces: authored by Particle-1 (see repo `distill/seeds`).

In [ ]:
!pip -q install --upgrade unsloth unsloth_zoo trl peft accelerate datasets bitsandbytes
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import json, glob, os
DATA = None
for pat in ['/kaggle/input/*/train.jsonl', '/kaggle/input/**/train.jsonl']:
    hits = glob.glob(pat, recursive=True)
    if hits: DATA = hits[0]; break
assert DATA, 'train.jsonl not found - add the dataset'
rows = [json.loads(l) for l in open(DATA) if l.strip()]
print(len(rows), 'examples'); print(rows[0]['messages'][1]['content'][:120])

In [ ]:
from unsloth import FastLanguageModel
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen3-1.7B',
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=1337,
    max_seq_length=max_seq_length,
)

In [ ]:
from datasets import Dataset
def fmt(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}
ds = Dataset.from_list(rows).map(fmt, remove_columns=[c for c in Dataset.from_list(rows).column_names if c != 'messages'])
print(ds[0]['text'][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        warmup_ratio=0.05, num_train_epochs=2, learning_rate=1e-4,
        logging_steps=10, save_strategy='no', optim='adamw_8bit',
        lr_scheduler_type='cosine', seed=1337, report_to='none',
        packing=False,
    ),
)
trainer.train()
trainer.save_model('/kaggle/working/adapter')

In [ ]:
# merge adapter into a plain HF checkpoint
model.save_pretrained_merged('/kaggle/working/merged', tokenizer, save_method='merged_16bit')
print('merged saved')

In [ ]:
# GGUF convert + Q4_0 quantize (pinned llama.cpp)
!git clone https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!cd /kaggle/working/llama.cpp && git checkout b54752df40e4d8e53d34ef5a598336b4e4871d39
!pip -q install -r /kaggle/working/llama.cpp/requirements.txt
!cd /kaggle/working/llama.cpp && cmake -B build -DGGML_NATIVE=ON -DGGML_OPENMP=ON && cmake --build build -j4 --target llama-quantize

In [ ]:
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/merged --outtype f16 --outfile /kaggle/working/model-f16.gguf
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/working/model-f16.gguf /kaggle/working/particle1-qwen3-1.7b-Q4_0.gguf Q4_0
!ls -lh /kaggle/working/*.gguf